# E2 — reward-hacking resistance via Feynman–Kac steering (self-contained, T4)

Optimizing a functional-proximity proxy in **open sequence space**: unconstrained proxy-greedy improves the
proxy by drifting OFF the protein manifold (naturalness craters — the reward-hacking your §hack result showed).
**FK/SMC steering** (proposals from the ESM masked-LM prior, reweighted by proxy × naturalness, resampled)
improves the proxy while staying natural. **Oracle-free** readout: naturalness + non-protein rate + proxy.

This is the *assay-independent* Phase-2 win (E1 was assay-dependent). **T4 → Run all (~10–20 min).**


### 1. Install


In [ ]:
!pip install -q transformers 2>/dev/null
import torch, numpy as np
print('cuda',torch.cuda.is_available())


### 2. Embedded data (SBS17a target profile + 477 protein CDS)


In [ ]:
import base64,gzip
_P=(
    "H4sIAJDKc2oC/4Waza5sOQ2Fx+e+C0f5dZIJUnMGNafPDCHEAIlJM+D9B3wrsbNL4jaoofveVbW3k9heXnbq49c//ZrH3//2+vPXP+v4eP/bjL/99tt6/phT"
    "/Pnf/7Ifv/zl64+//PWXj/SZUkkjFct15ZJq6YGVsVaeVsuc5WCll1qGzdRLX/Niglarc63q2FqW6mx5WW3ZbX3tzxKvzLmvkmu2UqqDvHHVVmbrvdg6IO9c"
    "zXLpreU86gPOXns1oDXinXzNZktjDqvLDb4+2uewZCmnUdvKvf3jD6kD6h1r8dGqY2QHc1sltZJXGaPVC84+Su/WWht9g10Gcx+dJbeRTKAb/P6wzzxZBHvv"
    "teZ2nhDYR5vWcqktLX/NGqv1ldeotaf8gCPlxn5kI0A9yf4KL7d6Db7cfVn/lNHrHKmmFliu1se0blYCW3MZx99z7XM8WOW53jj349KMsVSmDqLkMt2Wu6/b"
    "SoP9yMBKM0CeabmlshQZDvY2crKMqZk8qjaIL2rSC2o8PSbBY2NYm3O4vdf5MPeSMqdUCbBWRoBWs7w+2EgxB2tpq/dlirBUHnBob5m9zB6PL7M5Vye2CCQ3"
    "+P2ROemk7xNmCr19/ALzLJUlWCAsti58Ufok+h+w4eXJsVnvF8wdn/fGGsbjuu9wHSdNXHIE1jnPwLI1wjOvlRwi61rn1UaCjvJgpObixTMHxn545cg5xUl+"
    "u+cyzijsvCql+zmzbPy5JkUFke8RYTZYT08zD/ZzsYxv8SNeyv6+xJoJdo511eq2Xh/1c/VEjg7ogAAqnkkcViqsi2wmyzdYP1urMkwAkC71gvLW4hyNgHrL"
    "zpXwwCCR29s5fh+XDmKYqLLOf3npBW32rmVDRR5tI7PhVokcTGa7IPlC6Co08wxQ8QGpFLjNxCrflzJ/nnNJMTaJU14XzFYIVQ68JQJ/zAtCm6Q5a775AiXO"
    "Qfobnp9ruLlgzYUDOjDcyTcCtAHVwreT8AzQrEEo5Bn+snlBfFUJWr7bYmlwLVQl9maT2Q163hX9n2TumUCKZIJ5c7Nm+pOHAX/sk8zvlXNK095AtkYBIAxa"
    "gGap8R2SL8VxHv9lvZOFWeEIzIOzwdKQJpXGfHOqTyQZoThJhv6GEfd99WbxvSzzHGlTLm5TX3hufXIM8hF2OsS+g2t9doUf55hh7HLCcH5OorhNYqRAK4ER"
    "17Xx6qYCEGCa1A2YjSMrUQ5kb7uOICF9ksKSt40D2WJVS0GYDidRGKDKPmA0CsQIiDLAP62QMgciYUlNEi5bL27GHUasE2J4fcj1NUBOgd21TrXsHiHV2EEi"
    "I8Zg0Q8G5WfWSWKvACGGTt2B+pJ77Ms9RqRSOngMyiS3HVPJgWMS2zhOZPGU1KX6Bb+1N4yaPyAOjy0Ib1Fr2e1AjJwTjPrGGcFnvDFN4jV53E1KMqZwExHh"
    "FU6lt0ETZFaHFe0BiSzlkAjCQbKoi8GwVzzbbpEbev+scKWlOCEjGEkhow5b8UrJPieZTYkjUCyOTaBoxNbDlmnva8IDTUwb9l7BNPs4CNdUCZPLSeQLG2Gf"
    "rY5YWaVsEWqIh9zeQMkLnQ95EKANwoT6M/KYzQ1+nzjqUhecIKTq8cfbWK1JGKQTf0lFmirHZjmmByJNoZtx7GRCf2u5sj/58fUmK6UGVK7gDKjHYwQQmqDq"
    "mkqzr7WQzAQbfqP4jgcT1xPWLCvAIVnSFVJZlfvrTVmSLeKRRoggrELKcCg6HD4Y7kmIfo3G3wdVva6LUeKQe5SF1ONZygGUjibIw029PsrnVkfsSOViQA0U"
    "EUA0gooQPM/rN5g/YSrchQP5QJIgwAIDSjMTEQ6yIqQE75uE0Qbd4HfoGCUmZC+uCS1MuTfVWqoQUuwqHtzKMvBoRR28gYhPJDn/CxlUkal4nGehiLPDm3MQ"
    "KiE/OQwoPHsQwyycelliUVsh6fg7q5MaIbrfwL7kK9mNxyk+kIvqnhLdDXrOFYoWD3EO7bJHEgdmKLeqxNUaoBQLlUmsELVJ4IQVVfnKvKAMkq9DiVHc4Ot+"
    "qGBcqjwrzgSDk+KFqELNzZCbRF1XQlMviaQHrHqYHVqKx9HnEp9TvDDC4LdEUcJRKkdo+BT6Z0od4xocWOyCqk04NGnH9QGl0bb72wgwp06sLxqwsja4DX5f"
    "3oRzYayhRLIoCeL5OfEVOTBDERN2dVd2ErbUN1C00ii/M9IX3sT17KN1SsB0gyExi1IFnje1gC4dOVuOb0iYp5CYiAK1JVSC4ZpImHqtLKUzLCQm3Mep817M"
    "ua3jPhgTZxFh4p/pAkyvoCEl4VcUMXZF+0bLMdQ/2MU4c6RvVasW69TzUhtyoNtydUJRZmGsgtCdsTZCkrpO9QGdsV5Ol2ib23UPhpd4s+RkYBKnTf1Z3WH5"
    "CEvoQ+0C6+LE2EyAZccWFZYDsgCllXTaOUqPILwOYZrK3fNwTTpp5N9cbm67DFPwNrsaqtvzQLyEbEOW136CHUlSJBm0VT8vQegN/NyV+geC05bUDaeaYlcv"
    "Ly/q2bqWRil0eUNRNkkgKLIdqJA8W5YvFvRAUoU7YFwEDVGSlH0ZYWY7akACC3FIYR8+WhiqnrylS5yfWjwka/puWHnphYZUIPg6CmFIQ6tMdfrzPLeZr+Mj"
    "8Q51haYIU2fllEGkDG+QQ4Z/CVqkc4C68lHcGxrqtwjn5zlNQSg12Y/s6Ebqq5oJoh5q2I8TcBJzFGOSfR6k4U8pno58CiRvISvZ2A6yI6FqJLPcwPbJJJSw"
    "qRoxT74hs6lqmxbWOoUR0jGxfiEg12kmNgQXwQpUur2xPd2Y8hFicXS3IpeQcXQGyFHY+pwVCOoBd+CTtNtQ5SUZnNT870IjgOSAYgqZWzaCPNa8KO3pwTZw"
    "KxUfEUJId4i/R0MnYa2VQpjUuNCzqlqUvSUaC3VYt0Lv6tAfhZy2gtPcR72b23OSI9DV3mlcQll1rGqCIb3A21xmiJs5RuiQMrgejAQjbzQtiPcRFXgaWUEo"
    "uK2XNIGKOMRFRqSOJkhbKOC1uV2q/QYoebMnDwjM+QaqvaNA1RzYrusIPxH7Bt3eIbquLmRouie/ZcdOg7VEs178OnqedJaeUKG72O4888INxZ/VrE5KVe2u"
    "/Xi9qcOsqRC1DNU1PA3BhiY7UzHTUhCzFIga3xKdpCAoqmkIFH2oSr/alaJWvrgl9xhqTQlR1QtN/3qRg5fonH/FKIxt0+xBSqjMByN4NWqyaC7hYb5oe3CF"
    "bHq5MDyqQONG6SqiOwV779aOVNVAlVD+GJ/q4SAmNCAZfQq/QAX30nBgNG9N6SiPsGW73ly+XBe2T7iUXezajgjwSQntvxiTukA2BYZOFWVXUeF6QCNvcCIl"
    "3wKEXrVX/E0rcu09rRgHqEOXEm8xziDnid+qCZY2HzpjqslFLmJn5gfU5JRuR0o0QOQwnCL1wHrcoMtCBDlZ00ga/J/aFTaIRVMBFrdHvwRzsfLda97eTy1F"
    "0mgI18ekkPNi9WAk7RxuT8peMlbZStiaSy/AKZWotoASFsM+KXtODvlJ81UfUM2fammSst+PJ02El8LI3hz4uoMrKWLWsyMsdPY5ZOo/jBSLVmGnZ00m5Xt1"
    "m3oYDZV5O4cSoPhUk3qO2U4u3IGjZmr0huRoG65G1AKoAaAiVm+P6TCRbRLDKHg3trGppJX2upiqw2RvZFd1U1+/1ySp+aCjo7apz/PWh2LDTsk4darrDZSw"
    "VU+oGeIBm+Y8apIUnbtJer1LQnQQXMXL4eYS0JTG2mXdoq+jqFRNOabY5mJJ2pcDo5L5MFUjRR5Mcoxb0ojftjSnjS8a9fqIX40HEUVIsMASoC4lCF3qCKz6"
    "gI2NDqV7dkxjqn0roSlJRMkRhXxokpBqr6iv8RZKr+KXjoSEXw5iDi3KK6hxK7+B1JGhqYu3DjxOS76neMqv+mbwpB0fq7GHyGjWfbhS1cqr2JbpZVkTIpIl"
    "SdxXdT4Xk35EuzQXkJpDUW9YrdLVLb0iGXvS0E/1YKbyBlIhTWIvBqRDI2wact7KOb2DlEcoha6mPdPgriE69Y667AZPxmk+IAHDa9Q5OMbxVOlXAsG7PiIU"
    "BZGkQVr3PhpMeobyqYuMFs+SBENDn6rLrtcjFFVmqrQFUaSeN9r2InWHMuuFUhtghsSVl7aj7g00DQClnaMb1dUcXyTWdZfjBs+wsamJ0QVP1kXIgWjTOMZM"
    "8AxX0F0Df1xsu6hcSJdA2uMZXZMRs9K7g6tVczPHadASQY27usRIfzDTcUzdKHkoEF6mpk4kUR4M2Wm7lGQPj6kpIgpPosBNuSShdUBTo/ZXaz4WEBENEaf6"
    "R5/0EBOcBjGrTsH7R8VJUiGeuu7p/ix5xs4JDcOz29atbihpXerwiYZU0XshkNEfOIZHXfApinalILxwZXkDEdy1o8KjAjWEKEyoEq3rEjcYQw8NavCLBhEt"
    "5uRFsxCJPEK7unJMu6mbtek6o975xu70EBVDVc6rMaKKg+Rsy5YTbvD1v8qpwKFmm73GfEOzCSoWqoEO3tN9g31PD5FzEaVTnVjRPrsGym7Qx8S4j1cWPMyi"
    "jgvUeHRN0DXc86gmjXcbgAXN/S7GCSCZNPKb/ix+MbVlWVzy4/t93vizmYDAouQWu4cOl9CkJynSJyThOziHaE+7C3BJKoylm7rl9lTgVIKosJ0+qLr4Apxr"
    "z9pUB5pj6sXhG/gs+ZXixrJIHQHZeigFU0MsMdU0hdic/P1/ho5dX9bVEWwXk8S2pzumSfkcFxRvqv1RC+5P03mb6FGZ0E45/X6fOaq3rKx0QAlvY9mlIRHr"
    "libt0S8RJ3Kg7mPzG9Z3saBPuo2VhmVjD0gIgm0vso/+We0d9DoiWauaHP0uYV8AOKeIc6aSW8UvX0wT8M0WZ74gMmOl+qaaELcU8/3/HhQIpG7pHkrMuGLc"
    "xYaarvA0v7miUlPyvKmPzL3g0I0++rOrmrjBuJzR0E+RpCSJSzk4kFerCmjkEyBiha9IsUutX1ADU+pAt+iLNPPXoERrgaLD4PVdFSNrDMTrg0PmLqhF2hIK"
    "DZdIzw+NqfSThTdQBzjLkwoaIRiVP2kEYSf37qxRbaLppoFea93RsP9aBA5gtzlIVj9kGIh+XZb2B2R5ujlNyW+nlbLqm/STBQK1usEvtUUUagLfNjuv6JVQ"
    "R/yVPVo0VUt5oztWU61zCSZQl12a7rDJC1IpeY6FIXTWTb/v29rxcdGUmoBI+Y6GadhQRlU/dYhfF5BURBC1v9xsEAj/qw7plxMOopttqZdSb9rcnHuQ7kJX"
    "CewSX8bEf/eHMo8wuBJIXQvcSQihxOK+SSBP079SiqIno/Y2/V6Ac0arDAy+DR5396a2amr+NwPU8og+GMBGbBpIorjuqcA7mCRAtfDb+WrKJ/1RNT93gycL"
    "0Qu6+KJb0bTcBwYahEzxw6Ks+GDBNCnFFpqre8u5J7q7D8V/wx9VP6YuhsfbdFPeGZANumGd+llFzGLUtFC2dBMTEjPrnkAMDEGUBzKNejWRiFk1XR+sr/KL"
    "Z7Kb8lmxbszU+5uEjncXIgZKtcZZ6em6isShfiPj98IHIyLlm+nz+mz7eoWCLuZe29bX/d3Wkh21jopox3QV3tqeZCd/x/5Bku3BU/yGRtgZQetGPTACVLOV"
    "qh/ImNs6ylL3H0tTaP59YkxKQDK7aYx+RmzwNBSFuEz7p04Bdd3CxP0zb9J18tQzdGFu5PjpZ3dAYEMlSqeYXZurBVXmZF3iuwwRpsEH8UAYtMDkPP3iBS0b"
    "trajihoZXf/qcvAQjm6Iq+4E+bD5yHZrlf2TGfZqASGn9T3NMg4kcqk7eqkq24yq2u92wRriq6TphyhRoTVoFZlsur8gQnTqHpc4uWDZP/HQVCqN4mX7TVP+"
    "7B5UeTw0h5ZIv/SjqaEm2RxNGfYGamSbd3d4Xwk1iuL2bz/c3it+t7Lnzerw7kWh7vwQ/FW/a+ozJkIa41FnqLCw9JWZGjRRNPXTpPs9ntOPC5Tn061597ZJ"
    "TjeCpIPPddv+mZt+vUDg9egokA4UAlN7Wy+Esa7L4NvTKvc1CGu6pK0//gN16CiSVSkAAA=="
)
_F=(
    "..."
    "9nF8+w80LraZLf0HAA=="
)
open('SBS17a.txt','wb').write(gzip.decompress(base64.b64decode(_P)))
open('prot.fasta','wb').write(gzip.decompress(base64.b64decode(_F)))
print('CDS',open('prot.fasta').read().count('>'))


### 3. Helpers


In [ ]:
import random
CODON={};_b='TCAG';_a='FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG';_i=0
for a in _b:
    for b in _b:
        for c in _b: CODON[a+b+c]=_a[_i]; _i+=1
def translate(nt):
    nt=nt.upper().replace(' ',''); aa=''.join(CODON.get(nt[i:i+3],'X') for i in range(0,len(nt)-2,3))
    return aa.split('*')[0] if '*' in aa else aa
def load_profile(p):
    s={}
    for ln in open(p):
        q=ln.split()
        if len(q)>=2 and len(q[0])>=7 and q[0][1]=='[' and q[0][5]==']':
            try:s[q[0]]=float(q[1])
            except:pass
    return s
def mutate_cds(nt,prof):
    s=list(nt.upper())
    for i in range(len(s)-2):
        ctx=''.join(s[i:i+3])
        for sig,pr in prof.items():
            if ctx==sig[0]+sig[2]+sig[6] and random.random()<pr: s[i+1]=sig[4]; break
    return ''.join(s)
def accumulate(nt,prof,r):
    c=nt
    for _ in range(r): c=mutate_cds(c,prof)
    return c
def load_cds(fa,maxaa):
    out=[];seq=''
    def flush():
        nonlocal seq
        if seq:
            cds=seq.upper().replace(' ','')
            if len(cds)>=30:
                aa=translate(cds)
                if 20<=len(aa)<=maxaa: out.append(aa)
        seq=''
    for ln in open(fa):
        if ln.startswith('>'): flush()
        else: seq+=ln.strip()
    flush(); return out
print('helpers ready')


### 4. ESM-2 8M (MLM): embedding, naturalness (pseudo-LL), and on-manifold proposals


In [ ]:
from transformers import AutoTokenizer, EsmForMaskedLM
dev='cuda' if torch.cuda.is_available() else 'cpu'; MAXA=200
tok=AutoTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D')
mlm=EsmForMaskedLM.from_pretrained('facebook/esm2_t6_8M_UR50D').eval().to(dev)
AA='ACDEFGHIKLMNPQRSTVWY'; aa_ids=[tok.convert_tokens_to_ids(a) for a in AA]
MASK=tok.mask_token_id
@torch.no_grad()
def score(seqs,bs=32):
    '''returns naturalness[N] (mean per-residue log P(x_i|x), one forward) and pooled emb[N,D].'''
    nats=[]; embs=[]
    for s in range(0,len(seqs),bs):
        ch=[x[:MAXA] for x in seqs[s:s+bs]]
        e=tok(ch,return_tensors='pt',padding=True,add_special_tokens=True).to(dev)
        o=mlm(**e,output_hidden_states=True); lg=torch.log_softmax(o.logits,-1); H=o.hidden_states[-1]; m=e['attention_mask']
        ids=e['input_ids']
        for i in range(len(ch)):
            v=int(m[i].sum())-2
            if v<=0: nats.append(-20.0); embs.append(np.zeros(H.shape[-1],np.float32)); continue
            pos=torch.arange(1,v+1,device=dev)
            nats.append(float(lg[i,pos,ids[i,pos]].mean()))
            embs.append(H[i,1:v+1].mean(0).float().cpu().numpy())
    return np.array(nats,np.float32), np.array(embs,np.float32)
@torch.no_grad()
def propose_mlm(seqs,bs=32):
    '''on-manifold edit: mask one random position per seq, sample new AA from the ESM prior.'''
    out=[]
    for s in range(0,len(seqs),bs):
        ch=[x[:MAXA] for x in seqs[s:s+bs]]
        e=tok(ch,return_tensors='pt',padding=True,add_special_tokens=True).to(dev); ids=e['input_ids'].clone(); m=e['attention_mask']
        mp=[]
        for i in range(len(ch)):
            v=int(m[i].sum())-2; p=random.randint(1,max(1,v)); mp.append(p); ids[i,p]=MASK
        lg=mlm(input_ids=ids,attention_mask=m).logits
        for i in range(len(ch)):
            probs=torch.softmax(lg[i,mp[i],aa_ids],-1); a=AA[int(torch.multinomial(probs,1))]
            L=list(ch[i]); L[mp[i]-1]=a; out.append(''.join(L))
    return out
def mutate_rand(seq):
    L=list(seq); p=random.randint(0,len(L)-1); L[p]=random.choice(AA); return ''.join(L)
print('scorers ready on',dev)


### 5. Target region T (SBS17a-mutant centroid) + natural-protein floor


In [ ]:
random.seed(0); prof=load_profile('SBS17a.txt')
prots=load_cds('prot.fasta',MAXA)
random.shuffle(prots); prots=prots[:120]
# need CDS to apply the signature; re-derive from a fresh load keeping cds
def load_pairs(fa,maxaa):
    out=[];seq=''
    def fl():
        nonlocal seq
        if seq:
            cds=seq.upper().replace(' ','')
            if len(cds)>=30:
                aa=translate(cds)
                if 20<=len(aa)<=maxaa: out.append((cds,aa))
        seq=''
    for ln in open(fa):
        if ln.startswith('>'): fl()
        else: seq+=ln.strip()
    fl(); return out
pairs=load_pairs('prot.fasta',MAXA); random.shuffle(pairs); pairs=pairs[:120]
wt_aa=[aa for _,aa in pairs]
mut_aa=[translate(accumulate(cds,prof,8)) or 'A' for cds,_ in pairs]
_,mut_emb=score(mut_aa); T=mut_emb.mean(0); T=T/ (np.linalg.norm(T)+1e-9)
nat_wt,_=score(wt_aa); floor=float(nat_wt.mean()-2*nat_wt.std())
def proxy(emb): return (emb@T)/(np.linalg.norm(emb,axis=1)+1e-9)   # cosine to target region
print('natural floor (mean-2sd of WT naturalness):',round(floor,3),'| WT nat mean',round(float(nat_wt.mean()),3))


### 6. Two optimizers: unconstrained proxy-greedy vs Feynman–Kac steering


In [ ]:
STEPS=40; K=24; M=48; LAM=3.0
def greedy_free(wt,steps=STEPS,K=K):
    s=wt; tr=[]
    for t in range(steps):
        cands=[mutate_rand(s) for _ in range(K)]
        nat,emb=score(cands); px=proxy(emb); j=int(np.argmax(px)); s=cands[j]
        tr.append((float(px[j]),float(nat[j])))
    return s,tr
def fk_steer(wt,steps=STEPS,M=M,lam=LAM,tau=floor):
    part=[wt]*M; tr=[]
    for t in range(steps):
        prop=propose_mlm(part); nat,emb=score(prop); px=proxy(emb)
        z=(px-px.mean())/(px.std()+1e-6)
        w=np.exp(lam*z)*(nat>=tau)                     # FK potential: proxy AND on-manifold
        if w.sum()<=0: w=np.ones(M)
        w=w/w.sum(); idx=np.random.choice(M,M,p=w); part=[prop[i] for i in idx]
        tr.append((float(px.mean()),float(nat.mean()),float((nat<tau).mean())))
    return part,tr
print('optimizers defined')


### 7. Run both from several WT starts; track proxy, naturalness, non-protein rate


In [ ]:
N_START=8; starts=wt_aa[:N_START]
G=[]; F=[]; g_final=[]; f_final=[]
for si,wt in enumerate(starts):
    _,gtr=greedy_free(wt); G.append(gtr)
    fpart,ftr=fk_steer(wt); F.append(ftr)
    # final designs: greedy last seq vs FK particles
    gs=greedy_free(wt)[0]; gn,ge=score([gs]); g_final.append((float(proxy(ge)[0]),float(gn[0])))
    fn,fe=score(fpart); f_final.append((float(proxy(fe).mean()),float(fn.mean())))
    print(f'  start {si+1}/{N_START} done')
G=np.array(G); F=np.array([[*x] for tr in F for x in tr]).reshape(N_START,STEPS,3)
print('runs done')


### 8. Verdict + figure


In [ ]:
import matplotlib; import matplotlib.pyplot as plt, json
g_prox=G[:,:,0].mean(0); g_nat=G[:,:,1].mean(0)
f_prox=F[:,:,0].mean(0); f_nat=F[:,:,1].mean(0); f_bad=F[:,:,2].mean(0)
gf=np.array(g_final); ff=np.array(f_final)
g_nonprot=float(np.mean(gf[:,1]<floor)); f_nonprot=float(np.mean(ff[:,1]<floor))
print('final proxy   greedy %.3f   FK %.3f'%(gf[:,0].mean(),ff[:,0].mean()))
print('final naturaln greedy %.3f   FK %.3f   (floor %.3f)'%(gf[:,1].mean(),ff[:,1].mean(),floor))
print('non-protein rate  greedy %.0f%%   FK %.0f%%'%(100*g_nonprot,100*f_nonprot))
conf=(gf[:,1].mean()<floor) and (ff[:,1].mean()>=floor)
print('\nVERDICT:', 'CONFIRM — proxy-greedy reward-hacks off-manifold; FK steering stays natural while improving proxy' if conf
      else 'CHECK — inspect trajectories (tune LAM/tau)')
fig,ax=plt.subplots(1,3,figsize=(11,3.2))
ax[0].plot(g_prox,label='proxy-greedy',color='#d62728'); ax[0].plot(f_prox,label='FK steering',color='#1f77b4')
ax[0].set_title('proxy (cosine to target) ↑'); ax[0].set_xlabel('step'); ax[0].legend(fontsize=8)
ax[1].plot(g_nat,color='#d62728'); ax[1].plot(f_nat,color='#1f77b4'); ax[1].axhline(floor,ls='--',color='k',lw=.8,label='natural floor')
ax[1].set_title('naturalness (ESM pseudo-LL)'); ax[1].set_xlabel('step'); ax[1].legend(fontsize=8)
ax[2].scatter(gf[:,0],gf[:,1],color='#d62728',label='proxy-greedy'); ax[2].scatter(ff[:,0],ff[:,1],color='#1f77b4',label='FK steering')
ax[2].axhline(floor,ls='--',color='k',lw=.8); ax[2].set_xlabel('final proxy'); ax[2].set_ylabel('final naturalness'); ax[2].set_title('Pareto: proxy vs naturalness'); ax[2].legend(fontsize=8)
fig.tight_layout(); fig.savefig('e2_rewardhacking.png',dpi=160); print('saved e2_rewardhacking.png')
json.dump({'final_proxy_greedy':float(gf[:,0].mean()),'final_proxy_fk':float(ff[:,0].mean()),
           'final_nat_greedy':float(gf[:,1].mean()),'final_nat_fk':float(ff[:,1].mean()),'floor':floor,
           'nonprotein_greedy':g_nonprot,'nonprotein_fk':f_nonprot,'confirm':bool(conf)},
          open('e2.json','w'),indent=2); print('saved e2.json — send it + the png')
